In [0]:
%run ../../config/path

In [0]:
dbutils.widgets.text("catalog_name", "lending_risk_dev") # Default value for manual testing
CATALOG = dbutils.widgets.get("catalog_name")

print(f"Running setup for Catalog: {CATALOG}")

In [0]:
# # --- 1. Catalog Initialization --- 
print(f"Creating -- CATALOG => {CATALOG}")
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

# --- 2. Schema/databse Initialization ---
for schema in SCHEMAS: 
    schema_name = f"{CATALOG}.{schema}"    
    print(f"Creating -- SCHEMA => {schema_name}") 
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")    

In [0]:
# --- 3. Volume Initialization (The Landing & Checkpoint Zones) ---
print(f"Creating -- VOLUME => {LANDING_SCHEMA}.vol_landing")
spark.sql(f"""
            CREATE EXTERNAL VOLUME IF NOT EXISTS {LANDING_SCHEMA}.vol_landing
            LOCATION 'abfss://landing@extadlslendingriskdev.dfs.core.windows.net/'
            COMMENT 'Landing zone for raw files ingestion';        
          """)

print(f"Creating -- VOLUME => {BRONZE_SCHEMA}.vol_bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_SCHEMA}.vol_bronze")
 
print(f"Creating -- VOLUME => {SILVER_SCHEMA}.vol_silver")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SILVER_SCHEMA}.vol_silver")

print(f"Creating -- VOLUME => {GOLD_SCHEMA}.vol_gold")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {GOLD_SCHEMA}.vol_gold")

In [0]:
# --- 4. Table Initialization ---
# We use dbutils.notebook.run to execute the layer-specific DDL scripts
print("Building Bronze Layer Tables...")
dbutils.notebook.run("01_create_tables_bronze", 600)

print("Building Silver Layer Tables...")
dbutils.notebook.run("02_create_tables_silver", 600)

print("Building Gold Layer Tables...")
dbutils.notebook.run("03_create_tables_gold", 600)

print("--- LENDING RISK ENVIRONMENT READY ---")